# 문서 파싱 실습 — 26.09.29

이미 받은 파일로 실습합니다. **다운로드 코드가 없습니다.** 아래 `INPUT_DIR`에 문서 폴더를 지정한 뒤 위에서부터 실행하세요. 프로젝트 `.venv` 커널을 사용합니다.

1. 파일 분석
2. 파싱 불가 파일 제외
3. PDF 파싱
4. HWPX·DOCX 파싱
5. 문서별 문단 수·표 수·각 문단 첫 50글자 출력
6. 표 마크다운 변환 및 저장


In [15]:
from pathlib import Path
from collections import Counter
import sys
import hashlib
import unicodedata

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "backend/app/rag/local_parsers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("agent_practice 프로젝트 내부에서 실행하세요.")
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
from app.rag.local_parsers import HANDLERS, parse_local

DAY02 = ROOT / "sandbox/w5/day02"
# macOS의 한글 파일명 정규화 차이를 고려해 실제 폴더를 찾습니다.
INPUT_DIR = next((p for p in DAY02.iterdir()
                  if p.is_dir() and unicodedata.normalize("NFC", p.name) == "26.09.29_파싱실습"),
                 DAY02 / "26.09.29_파싱실습")
OUTPUT_DIR = DAY02 / "03_문서파싱실습_결과"
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"입력 폴더를 확인하세요: {INPUT_DIR}")
print("입력:", INPUT_DIR)
print("결과:", OUTPUT_DIR)

입력: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/26.09.29_파싱실습
결과: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/03_문서파싱실습_결과


### 파일 형식 분석

- 입력 폴더의 모든 파일을 탐색
- 확장자별 파일 개수를 집계
- `.docx`, `.pdf`, `.hwpx` 파일 수를 각각 확인
- 그 외 확장자는 `그 밖`으로 합산

In [23]:
# 1. 파일 분석
from collections import Counter

SAMPLES = INPUT_DIR
ORDER = [".docx", ".pdf", ".hwpx"]

counts = Counter(
    p.suffix.lower()
    for p in SAMPLES.rglob("*")
    if p.is_file()
)
others = sum(n for ext, n in counts.items() if ext not in ORDER)

print(f"파일 {sum(counts.values())}개")
print(
    " " + " , ".join(f"{ext} {counts[ext]}" for ext in ORDER)
    + f" , 그 밖 {others}"
)

파일 8개
 .docx 0 , .pdf 2 , .hwpx 4 , 그 밖 2


## 2. 파싱 불가한 파일 제외
기존 `local_parsers.py`의 `HANDLERS`를 확인해 지원하지 않는 확장자를 제외합니다. 기존 파서가 지원하는 PDF·HWPX·DOCX를 모두 처리합니다.

이 단계는 확장자와 빈 파일만 검사합니다. 파일 손상이나 추출할 텍스트가 없는 문제는 3·4단계에서 기존 `parse_local()`의 반환값과 예외로 판별합니다. 원본은 삭제하지 않습니다.

In [25]:
candidates = {ext: [] for ext in HANDLERS}
excluded = []
parsed_docs = {}  # 기존 파서가 반환한 ParsedDoc 객체를 그대로 보관

for path in files:
    ext = path.suffix.lower()
    reason = None
    if ext not in HANDLERS:
        reason = "기존 local_parsers.py에서 지원하지 않는 형식"
    elif path.stat().st_size == 0:
        reason = "빈 파일"
    if reason:
        excluded.append({"file": str(path.relative_to(INPUT_DIR)), "reason": reason})
        print("제외:", path.name, "→", reason)
        continue
    candidates[ext].append(path)

for ext, paths in candidates.items():
    print(f"{ext} 후보: {len(paths)}개")

제외: 서울시 초미세먼지 경보 현황.json → 기존 local_parsers.py에서 지원하지 않는 형식
제외: 형사사법절차 전자화 촉진법(법률)(제22008호)(20261002).hwp → 기존 local_parsers.py에서 지원하지 않는 형식
.docx 후보: 0개
.pdf 후보: 2개
.hwpx 후보: 4개


## 3. PDF 파싱
`parse_local(path)`가 확장자에 따라 기존 `parse_pdf()`를 호출합니다. 노트북에는 별도 PDF 파서를 만들지 않습니다.

기존 PDF 파서는 **텍스트가 있는 페이지 하나를 `조항` 블록 하나로** 반환합니다. 따라서 여기서 세는 본문 블록 수는 실제 문단 수가 아닌 텍스트가 있는 페이지 수입니다. **PDF 표 추출은 구현되어 있지 않아 `table_count=0`이며, 원본에 표가 없다는 의미가 아닙니다.** 빈 페이지는 기존 함수에서 건너뜁니다.

손상 파일 등은 `None`이 반환될 수 있고, 텍스트가 전혀 없으면 예외가 발생합니다. 두 경우 모두 제외 목록에 기록합니다.

In [32]:
# 3. pdf 파싱
# pdf 파일 순차적으로 가져오기

def parse_pdf(path: Path) -> list[str]:
    reader = PdfReader(str(path))
    return [(page.extract_text() or "").strip() for page in reader.pages]

pdf_files = sorted(SAMPLES.glob('*.pdf'))

print(f"PDF 파일 개수 : {len(pdf_files)}개")
for pdf_file in pdf_files:
    print(f"가져온 파일 : {pdf_file.name}")

PDF 파일 개수 : 2개
가져온 파일 : 경영지도사 및 기술지도사에 관한 법률(법률)(제22031호)(20260929).pdf
가져온 파일 : 독도 등 도서지역의 생태계 보전에 관한 특별법(법률)(제21065호)(20260926).pdf


## 4. HWPX·DOCX 파싱
`parse_local(path)`가 확장자에 따라 기존 `parse_hwpx()` 또는 `parse_docx()`를 호출합니다. 반환된 `ParsedDoc`을 `parsed_docs[path]`에 그대로 보관합니다.

기존 함수는 본문을 `조항`, 표를 `표` 블록으로 만듭니다. 표는 함수 내부에서 기존 `_table_to_markdown()`을 호출해 이미 마크다운으로 변환합니다. 노트북에서 변환 함수를 새로 만들거나 같은 표를 다시 변환하지 않습니다.

`page_count=1`은 기존 파서의 규약이며 실제 쪽수가 아닙니다. 중첩 표·병합 셀이나 표와 본문이 섞인 문단은 기존 파서의 추출 범위에 따릅니다. 백엔드 소스와 반환 객체를 수정하지 않습니다.

In [ ]:
# 4. hwpx 파싱
import zipfile

hwpx_files = sorted(SAMPLES.glob("*.hwpx"))
print(f"HWPX 파일 {len(hwpx_files)}개")

for HWPX in hwpx_files:
    with zipfile.ZipFile(HWPX) as zf:
        names = zf.namelist()

    print()
    print("가져온 파일: ", HWPX.name)

HWPX 파일 4개

가져온 파일:  가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).hwpx

가져온 파일:  공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).hwpx

가져온 파일:  시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).hwpx

가져온 파일:  형사소송법(법률)(제22009호)(20261002).hwpx


In [20]:
for path, doc in parsed_docs.items():
    paragraphs = [b for b in doc.blocks if b.kind == "조항"]
    tables = [b for b in doc.blocks if b.kind == "표"]
    label = "페이지 단위 본문 블록" if path.suffix.lower() == ".pdf" else "문단"
    print("=" * 80)
    print(path.name)
    print(f"결과: {type(doc).__name__} | {label} {len(paragraphs)}개 | 추출 표 {len(tables)}개")
    print(f"기존 ParsedDoc.table_count: {doc.table_count}")
    if path.suffix.lower() == ".pdf":
        print("참고: 기존 PDF 파서는 실제 문단 분리 및 표 추출을 지원하지 않습니다.")
    for i, block in enumerate(paragraphs, 1):
        print(f"{label} {i:03d} [{block.locator}]: {block.text[:50]}")

print(f"\n성공 {len(parsed_docs)}개 / 제외 {len(excluded)}개")
for item in excluded:
    print("제외:", item["file"], "→", item["reason"])
if len(parsed_docs) < 5:
    print("과제 조건 미충족: 성공 문서가 최소 5개 필요합니다.")

경영지도사 및 기술지도사에 관한 법률(법률)(제22031호)(20260929).pdf
결과: ParsedDoc | 페이지 단위 본문 블록 8개 | 추출 표 0개
기존 ParsedDoc.table_count: 0
참고: 기존 PDF 파서는 실제 문단 분리 및 표 추출을 지원하지 않습니다.
페이지 단위 본문 블록 001 [p.1]: 법제처                                               
페이지 단위 본문 블록 002 [p.2]: 법제처                                               
페이지 단위 본문 블록 003 [p.3]: 법제처                                               
페이지 단위 본문 블록 004 [p.4]: 법제처                                               
페이지 단위 본문 블록 005 [p.5]: 법제처                                               
페이지 단위 본문 블록 006 [p.6]: 법제처                                               
페이지 단위 본문 블록 007 [p.7]: 법제처                                               
페이지 단위 본문 블록 008 [p.8]: 법제처                                               
독도 등 도서지역의 생태계 보전에 관한 특별법(법률)(제21065호)(20260926).pdf
결과: ParsedDoc | 페이지 단위 본문 블록 6개 | 추출 표 0개
기존 ParsedDoc.table_count: 0
참고: 기존 PDF 파서는 실제 문단 분리 및 표 추출을 지원하지 않습니다.
페이지 단위 본

## 6. 기존 파서가 변환한 마크다운 표 저장
기존 `parse_hwpx()`와 `parse_docx()` 안에서 `_table_to_markdown()`으로 변환된 표의 `block.text`를 모읍니다. 문서별 `*_tables.md`와 전체 표 모음 `all_tables.md`만 저장합니다. PDF 결과에는 기존 파서의 표 추출 미지원 안내를 적습니다.

`ParsedDoc`은 메모리에서만 유지하고 통계와 미리보기는 화면에 출력합니다. 새 파싱·변환 함수를 정의하지 않습니다. `write_text()`가 있는 이 셀을 직접 실행할 때에만 MD 파일이 생성됩니다. 실행마다 새 결과 폴더를 사용합니다.

In [21]:
from datetime import datetime

if not parsed_docs:
    raise ValueError("성공한 문서가 없습니다. 입력 파일과 제외 사유를 확인하세요.")
RUN_DIR = OUTPUT_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
all_parts = ["# 파싱 문서의 표 모음"]
for path, doc in parsed_docs.items():
    relative = path.relative_to(INPUT_DIR).as_posix()
    suffix = hashlib.sha256(relative.encode()).hexdigest()[:8]
    output_name = f"{path.stem}_{path.suffix[1:].lower()}_{suffix}"
    parts = [f"# {path.name}", f"원본: {relative}"]
    tables = [b for b in doc.blocks if b.kind == "표"]
    for i, block in enumerate(tables, 1):
        parts.extend([f"## 표 {i} ({block.locator})", block.text])
    if path.suffix.lower() == ".pdf":
        parts.append("기존 PDF 파서는 표 추출을 지원하지 않습니다. 원문에 표가 없다는 뜻이 아닙니다.")
    elif not tables:
        parts.append("기존 파서에서 추출된 표가 없습니다.")
    markdown = "\n\n".join(parts) + "\n"
    (RUN_DIR / f"{output_name}_tables.md").write_text(markdown, encoding="utf-8")
    all_parts.append(markdown)
(RUN_DIR / "all_tables.md").write_text("\n\n---\n\n".join(all_parts), encoding="utf-8")
print("결과 폴더:", RUN_DIR)
print("성공 문서:", len(parsed_docs), "개")
print("전체 표:", sum(b.kind == "표" for doc in parsed_docs.values() for b in doc.blocks), "개")
print("최소 5개 조건:", "충족" if len(parsed_docs) >= 5 else "미충족")

결과 폴더: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/03_문서파싱실습_결과/20260929_170945_588215
성공 문서: 6 개
전체 표: 4 개
최소 5개 조건: 충족
